In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/raw/pha_dataset.csv')

print("Raw dataset loaded")
print("Shape:", df.shape)

C:\Users\surek\AppData\Local\Temp\ipykernel_13100\2674801829.py:1: DtypeWarning: Columns (3: pdes, 4: name, 5: prefix) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/pha_dataset.csv')


Raw dataset loaded
Shape: (958524, 45)


In [3]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Total columns with missing:", len(missing))
print("\nMissing values:")
print(missing)

Total columns with missing: 26

Missing values:
prefix            958506
name              936460
albedo            823421
diameter_sigma    822443
diameter          822315
sigma_per          19926
sigma_ad           19926
sigma_i            19922
sigma_q            19922
sigma_om           19922
sigma_ma           19922
sigma_n            19922
sigma_tp           19922
sigma_e            19922
sigma_a            19922
sigma_w            19922
pha                19921
moid               19921
H                   6263
moid_ld              127
neo                    4
ad                     4
per                    4
rms                    2
ma                     1
per_y                  1
dtype: int64


In [4]:
#Calculate Missing Percentage

missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

print("Missing percentage:")
print(missing_pct)

Missing percentage:
prefix            99.998122
name              97.698128
albedo            85.905100
diameter_sigma    85.803068
diameter          85.789714
sigma_per          2.078821
sigma_ad           2.078821
sigma_i            2.078404
sigma_q            2.078404
sigma_om           2.078404
sigma_ma           2.078404
sigma_n            2.078404
sigma_tp           2.078404
sigma_e            2.078404
sigma_a            2.078404
sigma_w            2.078404
pha                2.078300
moid               2.078300
H                  0.653400
moid_ld            0.013250
neo                0.000417
ad                 0.000417
per                0.000417
rms                0.000209
ma                 0.000104
per_y              0.000104
dtype: float64


In [5]:
#Drop High-Missing Columns (>50%)

drop_cols = ['prefix', 'name', 'albedo', 'diameter_sigma']

df = df.drop(columns=drop_cols)

print("Dropped columns:", drop_cols)
print("Shape after drop:", df.shape)

Dropped columns: ['prefix', 'name', 'albedo', 'diameter_sigma']
Shape after drop: (958524, 41)


In [6]:
#Drop Rows with Missing Target (PHA)

df = df.dropna(subset=['pha'])

print("Shape after dropping missing pha:", df.shape)

Shape after dropping missing pha: (938603, 41)


In [7]:
# Numeric columns - fill with median
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical columns - fill with mode
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handling:")
print(df.isnull().sum().sum())

C:\Users\surek\AppData\Local\Temp\ipykernel_13100\2979310397.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


Missing values after handling:
0


In [10]:
#duplicates

dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed")
    print("Shape after removing duplicates:", df.shape)

Duplicate rows: 0


In [15]:
print("Before cleaning:")
print("Shape:", df.shape)

# Eccentricity: 0 <= e < 1
df = df[(df['e'] >= 0) & (df['e'] < 1)]
print("\nAfter e filter:", df.shape)

# Inclination: 0 <= i <= 180
df = df[(df['i'] >= 0) & (df['i'] <= 180)]
print("After i filter:", df.shape)

# Semi-major axis: a > 0
df = df[df['a'] > 0]
print("After a filter:", df.shape)

# Diameter: > 0
df = df[df['diameter'] > 0]
print("After diameter filter:", df.shape)

Before cleaning:
Shape: (938599, 41)

After e filter: (938599, 41)
After i filter: (938599, 41)
After a filter: (938599, 41)
After diameter filter: (938599, 41)


In [16]:
#Remove Physically Impossible Values
# Eccentricity should be 0 to 1 (0 = circular, 1 = parabolic)
df = df[(df['e'] >= 0) & (df['e'] < 1)]

# Inclination should be 0 to 180
df = df[(df['i'] >= 0) & (df['i'] <= 180)]

# Semi-major axis should be positive
df = df[df['a'] > 0]

# Diameter should be positive (if exists)
if 'diameter' in df.columns:
    df = df[df['diameter'] > 0]

print("Shape after removing invalid values:", df.shape)


Shape after removing invalid values: (938599, 41)


In [17]:
print("CLEANED DATASET SUMMARY")

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")


CLEANED DATASET SUMMARY
Total rows: 938599
Total columns: 41
Missing values: 0
Duplicate rows: 0


In [18]:
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print("Saved")

Saved
